In [58]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path
from shapely.geometry import Point
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, mean_squared_error

In [59]:
# ---------------------------------------------------------
# Parameters & Set Up
# ---------------------------------------------------------
SPECIES = 'euplagia_quadripunctaria'
TARGET_REGION = "East"

TRAIN_START = 1971
TRAIN_END = 2022
VALIDATION_START = 2023
VALIDATION_END = 2025
FORECAST_YEAR = 2026

ABSENCE_RATIO = 3
RANDOM_SEED = 42

DATA_DIR = Path("/Users/alexrabeau/Desktop/SPHERE/NBN-LPI-forecasting/data/NBN-atlas/species/")
GRID_DIR = Path("/Users/alexrabeau/Desktop/SPHERE/NBN-LPI-forecasting/data/env_predictors/gridded/")
OUTPUT_DIR = Path("/Users/alexrabeau/Desktop/SPHERE/NBN-LPI-forecasting/submission/")

In [60]:
# ---------------------------------------------------------
# Load occurrence data
# ---------------------------------------------------------
species_dfs = {
    'euplagia_quadripunctaria': pd.read_csv(DATA_DIR / "Euplagia_quadripunctaria/euplagia_quadripunctaria_processed.csv"),
    'fratercula_arctica': pd.read_csv(DATA_DIR / "Fratercula_arctica/fratercula_arctica_processed.csv"),
    'haliaeetus_albicilla': pd.read_csv(DATA_DIR / "Haliaeetus_albicilla/haliaeetus_albicilla_processed.csv"),
    'lutra_lutra': pd.read_csv(DATA_DIR / "Lutra_lutra/lutra_lutra_processed.csv"),
    'pandion_haliaetus': pd.read_csv(DATA_DIR / "Pandion_haliaetus/pandion_haliaetus_processed.csv"),
    'pipistrellus_pipistrellus': pd.read_csv(DATA_DIR / "Pipistrellus_pipistrellus/pipistrellus_pipistrellus_processed.csv"),
    'sciurus_vulgaris': pd.read_csv(DATA_DIR / "Sciurus_vulgaris/sciurus_vulgaris_processed.csv"),
    'triturus_cristatus': pd.read_csv(DATA_DIR / "Triturus_cristatus/triturus_cristatus_processed.csv"),
    'vipera_berus': pd.read_csv(DATA_DIR / "Vipera_berus/vipera_berus_processed.csv")
}

# ---------------------------------------------------------
# Prepare occurrences
# ---------------------------------------------------------
occ = species_dfs[SPECIES].copy()
print("Occurrences:", occ.shape)

occ["Event.Date"] = pd.to_datetime(
    occ["Event.Date"],
    errors="coerce"
)

Occurrences: (8275, 37)


In [61]:
# ---------------------------------------------------------
# Define Environmental Predictors
# ---------------------------------------------------------

NON_PREDICTORS = {
    "Occurrence.ID",
    "Scientific.name",
    "Event.Date",
    "Day",
    "Month",
    "Year",
    "Latitude..WGS84.",
    "Longitude..WGS84.",
    "grid_id",
    "Region"
}

predictors = [
    c for c in occ.columns
    if (
        c not in NON_PREDICTORS
        and c in occ.columns
        and pd.api.types.is_numeric_dtype(occ[c])
    )
]

print("Number of predictors:", len(predictors))


Number of predictors: 27


In [62]:
# ---------------------------------------------------------
# Create Presence data
# ---------------------------------------------------------

def get_presence_data(
    occurrences,
    year
):

    if len(occurrences) == 0:
        return None

    columns = [
        "grid_id",
        "Region",
        "Longitude..WGS84.",
        "Latitude..WGS84."
    ] + predictors

    presence_data = occurrences[
        columns
    ].copy()

    presence_data["presence"] = 1
    presence_data["year"] = year

    return presence_data

In [63]:
# ---------------------------------------------------------
# Generate Pseudo-absences
# ---------------------------------------------------------

def generate_pseudo_absences(grid, presence_grid_ids, n_absences, rng):

    # Remove cells containing known presences
    available = grid[
        ~grid["grid_id"].isin(
            presence_grid_ids
        )
    ].copy()

    if len(available) < n_absences:

        raise ValueError(
            "Not enough available grid cells "
            "to generate pseudo-absences."
        )

    sampled = available.sample(
        n=n_absences,
        random_state=rng
    ).copy()

    sampled["presence"] = 0

    return sampled


In [64]:
# ---------------------------------------------------------
# Dataset Construction
# ---------------------------------------------------------
def create_lagged_year_dataset(
    occurrence_year,
    occurrences,
    n_pa_per_presence,
    rng
):
    """
    Pairs occurrence_year occurrences with (occurrence_year - 1) environmental grid data.
    """
    env_year = occurrence_year - 1
    file = os.path.join(GRID_DIR, f"gridden_env_{env_year}.csv")
    
    if not os.path.exists(file):
        print(f"  Environmental file for year {env_year} not found.")
        return None
        
    grid = pd.read_csv(file)

    # Extract presences
    presence_data = get_presence_data(
        occurrences=occurrences,
        year=occurrence_year
    )

    if presence_data is None or len(presence_data) == 0:
        return None

    # Sample pseudo-absences from grid (env_year)
    presence_grid_ids = presence_data["grid_id"].dropna().unique()
    n_presences = len(presence_grid_ids)
    n_absences = n_presences * ABSENCE_RATIO

    absence_data = generate_pseudo_absences(
        grid=grid,
        presence_grid_ids=presence_grid_ids,
        n_absences=n_absences,
        rng=rng
    )
    absence_data["year"] = occurrence_year

    columns = [
        "grid_id",
        "Region",
        "Longitude..WGS84.",
        "Latitude..WGS84."
    ] + predictors + [
        "presence",
        "year"
    ]

    presence_data = presence_data[columns]
    absence_data = absence_data[columns]

    result = pd.concat([presence_data, absence_data], ignore_index=True)
    return result

In [65]:
# ---------------------------------------------------------
# 1. Model Training 
# ---------------------------------------------------------
rng = np.random.RandomState(RANDOM_SEED)
print("\nBuilding training data (Lag-1)...")

training_parts = []
for year in range(TRAIN_START, TRAIN_END + 1):
    yearly_occurrences = occ[
        (occ["Year"] == year) & (occ["Region"] != TARGET_REGION)
    ].copy()

    if len(yearly_occurrences) == 0:
        continue

    yearly_data = create_lagged_year_dataset(
        occurrence_year=year,
        occurrences=yearly_occurrences,
        n_pa_per_presence=ABSENCE_RATIO,
        rng=rng
    )

    if yearly_data is not None:
        training_parts.append(yearly_data)

train = pd.concat(training_parts, ignore_index=True)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=RANDOM_SEED
    )
)

X_train = train[predictors]
y_train = train["presence"]
model.fit(X_train, y_train)
print("Model fitted successfully.")


Building training data (Lag-1)...
Model fitted successfully.


In [67]:
# ---------------------------------------------------------
# 2. Model Validation & Evaluation
# ---------------------------------------------------------
print("\nBuilding validation data...")
validation_parts = []

for year in range(VALIDATION_START, VALIDATION_END + 1):
    yearly_occurrences = occ[
        (occ["Year"] == year) & (occ["Region"] == TARGET_REGION)
    ].copy()

    if len(yearly_occurrences) == 0:
        continue

    yearly_data = create_lagged_year_dataset(
        occurrence_year=year,
        occurrences=yearly_occurrences,
        n_pa_per_presence=ABSENCE_RATIO,
        rng=rng
    )

    if yearly_data is not None:
        validation_parts.append(yearly_data)

validation = pd.concat(validation_parts, ignore_index=True)

X_val = validation[predictors]
y_val = validation["presence"]

val_probability = model.predict_proba(X_val)[:, 1]

# Calculate Validation Metrics
auc = roc_auc_score(y_val, val_probability)
mse = mean_squared_error(y_val, val_probability)
rmse = np.sqrt(mse)

print("\n=============================")
print("VALIDATION RESULTS")
print("=============================")
print(f"Species:       {SPECIES}")
print(f"Target region: {TARGET_REGION}")
print(f"Training:      {TRAIN_START}-{TRAIN_END}")
print(f"Validation:    {VALIDATION_START}-{VALIDATION_END}")
print(f"AUC:           {auc:.3f}")
print(f"MSE (Brier):   {mse:.3f}")
print(f"RMSE:          {rmse:.3f}")

# Save validation output
validation_output = validation[[
    "year", "grid_id", "Region", "Longitude..WGS84.", "Latitude..WGS84.", "presence"
]].copy()
validation_output["prediction"] = val_probability
validation_output.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)


Building validation data...

VALIDATION RESULTS
Species:       euplagia_quadripunctaria
Target region: East
Training:      1971-2022
Validation:    2023-2025
AUC:           0.829
MSE (Brier):   0.192
RMSE:          0.438


In [68]:
# ---------------------------------------------------------
# 3. Generate 2026 Forecast
# ---------------------------------------------------------
print(f"\nGenerating {FORECAST_YEAR} Forecast using {FORECAST_YEAR - 1} Environmental Data...")

# Load 2025 gridded environmental data to forecast 2026 occurrences
grid_2025_file = GRID_DIR / f"gridden_env_{FORECAST_YEAR - 1}.csv"
grid_2025 = pd.read_csv(grid_2025_file)

# Extract predictor features for full grid prediction
X_forecast_2026 = grid_2025[predictors]

# Predict probability of occurrence for 2026
grid_2025["occurrence_probability_2026"] = model.predict_proba(X_forecast_2026)[:, 1]

# Save 2026 forecast predictions
forecast_cols = [
    "grid_id", "Region", "Longitude..WGS84.", "Latitude..WGS84.", "occurrence_probability_2026"
]
forecast_2026 = grid_2025[forecast_cols].copy()
forecast_2026["forecast_year"] = FORECAST_YEAR

output_path = OUTPUT_DIR / f"forecast_{SPECIES}_{FORECAST_YEAR}.csv"
forecast_2026.to_csv(output_path, index=False)
print(f"2026 Forecast saved to: {output_path}")


Generating 2026 Forecast using 2025 Environmental Data...
2026 Forecast saved to: /Users/alexrabeau/Desktop/SPHERE/NBN-LPI-forecasting/submission/forecast_euplagia_quadripunctaria_2026.csv
